In [1]:
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from tslearn.datasets import UCR_UEA_datasets
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [2]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [4]:
X_train.shape

(2459, 36, 6)

In [4]:
X_train_tensor = torch.tensor(X_train, dtype=torch.bfloat16)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.bfloat16)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [5]:
batch_size = 16
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=True)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import time
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

MODEL_PARAMS = {
    "patch_size": 16,  # 8, 16, 32, 64, 128
    "num_samples": 100,
    "target_dim": 1,
    "feat_dynamic_real_dim": 0,
    "past_feat_dynamic_real_dim": 0,
}

In [ ]:
from encoder import MoiraiEncoder

encoder = MoiraiEncoder(
    module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-small"),
    prediction_length=0,
    context_length=36,
    **MODEL_PARAMS,
)

In [13]:
import numpy as np
import torch


def preprocess_batched_multivariate(
    data: np.ndarray,
    *,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
):
    """
    data: np.ndarray of shape (B, T, V) = (n_individual, time, variate)
    Assumes NO missing values and NO padding in the raw data.
    """

    if not isinstance(data, np.ndarray):
        raise TypeError("data must be a numpy.ndarray")
    if data.ndim != 3:
        raise ValueError(f"Expected shape (B,T,V), got {data.shape}")

    B, T, V = data.shape

    # (B,T,V)
    past_target = torch.as_tensor(data, dtype=dtype, device=device)

    # observed mask: (B,T,V) all True since you said no missing values
    past_observed_target = torch.ones((B, T, V), dtype=torch.bool, device=device)

    # padding mask: (B,T) all False since you said no padding
    past_is_pad = torch.zeros((B, T), dtype=torch.bool, device=device)

    return past_target, past_observed_target, past_is_pad

In [25]:
target_tensor, is_target_observed, is_target_padded = preprocess_batched_multivariate(
    X_train[:16, :, :]
)

print(target_tensor.shape)  # (2459, 36, 6)
print(is_target_observed.shape)  # (2459, 36, 6)
print(is_target_padded.shape)  # (2459, 36)

torch.Size([16, 36, 6])
torch.Size([16, 36, 6])
torch.Size([16, 36])


In [24]:
X_train[:16, :, :].shape

(16, 36, 6)

In [20]:
reprs = encoder(
    past_target=target_tensor,
    past_observed_target=is_target_observed,
    past_is_pad=is_target_padded,
    feat_dynamic_real=None,
    observed_feat_dynamic_real=None,
    past_feat_dynamic_real=None,
    past_observed_feat_dynamic_real=None,
    num_samples=None,
)

In [21]:
reprs.shape

torch.Size([16, 18, 384])

In [ ]:
# (batch, time, variate)

In [12]:
X_train.shape, type(X_train)

((2459, 36, 6), numpy.ndarray)